In [ ]:
!pip install -q google-genai

In [ ]:
#Tokenization
from google import genai

client = genai.Client(api_key="API_KEY_HERE")

with open("Saudi Projects & Vision 2030.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Chunking with 20% character overlap
words = text.split()

chunk_size = 500
overlap = 100
step = chunk_size - overlap

chunks = []

for i in range(0, len(text), step):
    chunk = text[i:i + chunk_size]

    if chunk:
        chunks.append(chunk)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Characters: {len(chunk)}")
    print(chunk)


Number of chunks: 25

--- Chunk 1 ---
Characters: 500
Saudi Projects & Vision 2030
1. NEOM: A New Region for Innovation and Development

NEOM is a major development region in northwestern Saudi Arabia and one of the most prominent projects associated with Saudi Vision 2030. The project aims to develop new communities and economic sectors by combining technology, sustainability, innovation, and advanced infrastructure. NEOM covers a large area that includes coastal, mountainous, and desert environments along the Red Sea. Its development includes sev

--- Chunk 2 ---
Characters: 500
cludes coastal, mountainous, and desert environments along the Red Sea. Its development includes several specialized destinations, each designed around different economic and lifestyle concepts. NEOM is intended to support sectors such as tourism, technology, clean energy, advanced manufacturing, and entertainment. The project represents Saudi Arabia’s efforts to diversify its economy beyond traditional sourc

In [ ]:
#Tokenization
for i, chunk in enumerate(chunks):
    result = client.models.count_tokens(
        model="gemini-3.5-flash-lite",
        contents=chunk
    )

    print(f"Chunk {i+1}: {result.total_tokens} tokens")

Chunk 1: 98 tokens
Chunk 2: 82 tokens
Chunk 3: 88 tokens
Chunk 4: 75 tokens
Chunk 5: 93 tokens
Chunk 6: 90 tokens
Chunk 7: 84 tokens
Chunk 8: 84 tokens
Chunk 9: 83 tokens
Chunk 10: 82 tokens
Chunk 11: 90 tokens
Chunk 12: 80 tokens
Chunk 13: 91 tokens
Chunk 14: 87 tokens
Chunk 15: 97 tokens
Chunk 16: 95 tokens
Chunk 17: 88 tokens
Chunk 18: 85 tokens
Chunk 19: 77 tokens
Chunk 20: 93 tokens
Chunk 21: 85 tokens
Chunk 22: 100 tokens
Chunk 23: 99 tokens
Chunk 24: 77 tokens
Chunk 25: 39 tokens


In [ ]:
#Embeddings
embeddings = []

for chunk in chunks:

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )

    embeddings.append(result.embeddings[0].values)

print("Number of embeddings:", len(embeddings))


Number of embeddings: 25


In [ ]:
#Victors
for i, (chunk, vector) in enumerate(zip(chunks, embeddings)):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

    print("\nVector:")
    print(vector)


--- Chunk 1 ---
Saudi Projects & Vision 2030
1. NEOM: A New Region for Innovation and Development

NEOM is a major development region in northwestern Saudi Arabia and one of the most prominent projects associated with Saudi Vision 2030. The project aims to develop new communities and economic sectors by combining technology, sustainability, innovation, and advanced infrastructure. NEOM covers a large area that includes coastal, mountainous, and desert environments along the Red Sea. Its development includes sev

Vector:
[-0.0020062525, -0.019274076, -0.029477382, -0.059510738, -0.0037812942, -0.00879856, -0.0032633625, -0.00021297658, -0.024477717, -0.020481948, -0.016123202, 0.010608799, 0.010297204, 0.026395375, 0.13151959, 0.0045494246, 0.015114691, -0.01878202, -0.0071324822, 0.016736945, -0.021321446, -0.0045614303, 0.019076541, -0.009480702, -0.0031738188, -0.010777399, -0.003644547, -0.012121737, 0.024411744, 0.004585783, -0.005719123, -0.024264334, -0.013122573, 0.017805193, 0

In [ ]:
#Semantic Similarity Search + Answer
from sklearn.metrics.pairwise import cosine_similarity

def ask_question(question, top_k=3):

    # Convert the question into an embedding
    query_result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    query_vector = query_result.embeddings[0].values

    # Calculate cosine similarity
    similarities = cosine_similarity(
        [query_vector],
        embeddings
    )[0]

    # Get top 3 chunks
    top_indices = similarities.argsort()[-top_k:][::-1]

    print("\nQuestion:")
    print(question)

    print("\nTop 3 Most Relevant Chunks:")

    for rank, index in enumerate(top_indices, 1):
        print(f"\n--- Rank {rank} ---")
        print(f"Similarity Score: {similarities[index]:.4f}")
        print(f"Chunk {index + 1}:")
        print(chunks[index])

    # Combine retrieved chunks
    context = "\n\n".join(
        chunks[index] for index in top_indices
    )

    # Generate final answer
    prompt = f"""
Answer the question using ONLY the information in the context.

Context:
{context}

Question:
{question}

If the answer is not available in the context, say:
"I don't have enough information to answer this."
"""

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=prompt
    )

    print("\nAnswer:")
    print(response.text)

In [ ]:
#Question
ask_question("What is The Line?")


Question:
What is The Line?

Top 3 Most Relevant Chunks:

--- Rank 1 ---
Similarity Score: 0.6963
Chunk 4:
 a linear city without traditional roads and cars within its main urban environment. The development is intended to prioritize walking, accessibility, technology, and efficient use of land and resources. The Line is planned to incorporate advanced infrastructure and renewable energy while reducing the environmental footprint associated with conventional cities. Its design has attracted global attention because it challenges traditional approaches to urban planning and transportation. The project

--- Rank 2 ---
Similarity Score: 0.6664
Chunk 5:
ntion because it challenges traditional approaches to urban planning and transportation. The project is part of Saudi Arabia’s broader effort to develop future-oriented cities and introduce new models of sustainable urban development. The Line therefore represents the combination of architecture, technology, transportation, environmental p